In [2]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from transformers import AutoTokenizer
import os

In [3]:
# GLOBAL VARIABLES DECLARE HERE

MAX_TARGET_LEN = 512

# DECIDING MAX SEQ LEN

In [4]:
df = pd.read_csv("model1_ready_data.csv")

input_lengths = df['input_text'].str.len()
target_lengths = df['target_text'].str.len()

max_input = input_lengths.max()
max_target = target_lengths.max()

print(f"Max Input Length: {max_input}")
print(f"Max Target Length: {max_target}")

# To see the distribution (useful to see if a few outliers are skewing the max)
print(target_lengths.describe())

Max Input Length: 2099
Max Target Length: 5812
count    96108.000000
mean       267.518032
std         65.159268
min         31.000000
25%        235.000000
50%        256.000000
75%        284.000000
max       5812.000000
Name: target_text, dtype: float64


**As we can see the average length as in the average number of characters for any sequence is 268 and the 75th percentile is 284. But there is a max length of 5812. So we definitely should not just blindly pick the max as the seq length.**

**Instead we will choose a limit like 350 so as to include longer sequences anyway. But if we keep more it will just cause our model to unnecessarily slow down in processing.**

In [5]:
MAX_LIMIT = 350 

df_filtered = df[
    (df['input_text'].str.len() <= MAX_LIMIT) & 
    (df['target_text'].str.len() <= MAX_LIMIT)
].copy()

print(f"Original size: {len(df)}")
print(f"New size: {len(df_filtered)}")
print(f"Percentage kept: {len(df_filtered)/len(df)*100:.2f}%")

Original size: 96108
New size: 86712
Percentage kept: 90.22%


The removed data can be used for testing later.

# CONVERTING CHARACTERS TO INT IDs AND MAKING VOCABULARY

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
print(df_filtered.iloc[3]['target_text'])

sugata <p=n,c=2> | sa <p=i,c=nan> | suta <p=n,c=2> | sa <p=i,c=nan> | DarmakAya <p=n,c=2> | praRipat <p=vi,c=nan> | Adara <p=n,c=5> | aKila <p=a,c=2> | ca <p=i,c=nan> | vand <p=va,c=2> | sugata <p=n,c=N/A> + Atmaja <p=n,c=N/A> + saMvara <p=n,c=N/A> + avatAra <p=n,c=2> | kaTay <p=v,c=nan> | yaTAgama <p=a,c=2> | samAsa <p=n,c=5>


In [8]:
df_filtered.columns

Index(['input_text', 'target_text'], dtype='object')

In [9]:
df_filtered.shape

(86712, 2)

In [10]:
df_filtered.iloc[3]['input_text']

'sugatAn sa sutAn sa DarmakAyAn praRipatya AdarataH aKilAn ca vandyAn sugataAtmajasaMvaraavatAram kaTayizyAmi yaTAgamam samAsAt'

In [11]:
import torch
import torch.nn as nn

# 1. Define the alphabet and symbols, this will be the vocabulary
chars = " abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789'|+-<>={},"

# 2. Build char_to_ix with the i+4 offset
char_to_ix = {ch: i+4 for i, ch in enumerate(chars)}
char_to_ix["<PAD>"] = 0
char_to_ix["<UNK>"] = 1
char_to_ix["<SOS>"] = 2
char_to_ix["<EOS>"] = 3

# 3. Build the reverse mapping (ix_to_char)
ix_to_char = {i: ch for ch, i in char_to_ix.items()}

vocab_size = len(char_to_ix)

print(f"Vocab Size: {vocab_size}")
print(f"First few mappings: {list(char_to_ix.items())[:10]}")
print(f"Index 2: {ix_to_char[2]}")  # Should be <SOS>
print(f"Index 4: {ix_to_char[4]}")  # Should be ' ' (the first char in your string)

Vocab Size: 77
First few mappings: [(' ', 4), ('a', 5), ('b', 6), ('c', 7), ('d', 8), ('e', 9), ('f', 10), ('g', 11), ('h', 12), ('i', 13)]
Index 2: <SOS>
Index 4:  


In [12]:
import pandas as pd
import numpy as np

"""
PLEASE NOTE WE ARE USING THE FILTERED DATA WITH SEQ LEN MAX 350
"""

df = df_filtered

def encode_sequence(text, mapping, add_sos=False, add_eos=False):
    """
    Converts a string to a list of integer IDs.
    """
    # Start with SOS if requested
    tokens = [mapping["<SOS>"]] if add_sos else []
    
    # Map characters to IDs, defaulting to <UNK> if char not found
    for char in str(text):
        tokens.append(mapping.get(char, mapping["<UNK>"]))
        
    # Append EOS if requested
    if add_eos:
        tokens.append(mapping["<EOS>"])
        
    # Return as a string of integers separated by space
    return " ".join(map(str, tokens))

# --- Processing ---

print("Encoding input sequences...")
# Input text doesn't need SOS/EOS, just the raw characters
df['input_ids'] = df['input_text'].apply(
    lambda x: encode_sequence(x, char_to_ix, add_sos=False, add_eos=False)
)

print("Encoding target sequences...")
# Target text needs SOS for the decoder and EOS to know when to stop
df['target_ids'] = df['target_text'].apply(
    lambda x: encode_sequence(x, char_to_ix, add_sos=True, add_eos=True)
)


# We only save the ID columns
output_df = df[['input_ids', 'target_ids']]
output_filename = "icarus_preprocessed_indices.csv"
output_df.to_csv(output_filename, index=False)

print(f"Success! Saved to {output_filename}")

# --- Quick Test / Verification ---
sample_row = output_df.iloc[0]
print("\nSample Check:")
print(f"Input IDs (first 5): {sample_row['input_ids'].split()[:5]}")
print(f"Target IDs (first 5): {sample_row['target_ids'].split()[:5]}")

Encoding input sequences...
Encoding target sequences...
Success! Saved to icarus_preprocessed_indices.csv

Sample Check:
Input IDs (first 5): ['13', '29', '5', '17', '4']
Target IDs (first 5): ['2', '13', '8', '5', '17']


# MODEL ARCHITECTURE

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hid_dim, n_layers, dropout=dropout, bidirectional=True)
        self.fc = nn.Linear(hid_dim * 2, hid_dim) # To compress bidirectional states
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # src: [src_len, batch_size]
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        # outputs: [src_len, batch_size, hid_dim * 2]
        # hidden: [n_layers * 2, batch_size, hid_dim]
        
        # Concat the bidirectional hidden states
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        cell = torch.tanh(self.fc(torch.cat((cell[-2,:,:], cell[-1,:,:]), dim=1)))
        
        return outputs, (hidden.unsqueeze(0), cell.unsqueeze(0))

class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear((hid_dim * 2) + hid_dim, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        # hidden: [1, batch_size, hid_dim]
        # encoder_outputs: [src_len, batch_size, hid_dim * 2]
        src_len = encoder_outputs.shape[0]
        h = hidden.repeat(src_len, 1, 1)
        energy = torch.tanh(self.attn(torch.cat((h, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return F.softmax(attention, dim=0)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, hid_dim, n_layers, dropout, attention):
        super().__init__()
        self.attention = attention
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.rnn = nn.LSTM((hid_dim * 2) + emb_dim, hid_dim, n_layers, dropout=dropout)
        self.out = nn.Linear((hid_dim * 2) + hid_dim + emb_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs):
        # input: [batch_size]
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        
        # Calculate attention weights
        a = self.attention(hidden, encoder_outputs).unsqueeze(1)
        # a: [batch_size, 1, src_len]
        a = a.permute(2, 1, 0)
        
        encoder_outputs = encoder_outputs.permute(1, 0, 2)
        # print(encoder_outputs.shape)
        # print(a.shape)
        weighted = torch.bmm(a, encoder_outputs).permute(1, 0, 2)
        # weighted: [1, batch_size, hid_dim * 2]
        
        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        
        prediction = self.out(torch.cat((output, weighted, embedded), dim=2)).squeeze(0)
        return prediction, hidden, cell

# DATA FOR FINAL TRAINING

In [14]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class SanskritDataset(Dataset):
    def __init__(self, df):
        self.df = df
        # Convert the space-separated strings back to lists of ints
        self.inputs = self.df['input_ids'].apply(lambda x: [int(i) for i in x.split()]).values
        self.targets = self.df['target_ids'].apply(lambda x: [int(i) for i in x.split()]).values

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return torch.tensor(self.inputs[idx]), torch.tensor(self.targets[idx])

def collate_fn(batch):
    inputs, targets = zip(*batch)
    # Pad to the max length of THIS batch (dynamic padding)
    inputs_padded = pad_sequence(inputs, batch_first=True, padding_value=0)
    targets_padded = pad_sequence(targets, batch_first=True, padding_value=0)
    return inputs_padded, targets_padded

# Initialize
dataset = SanskritDataset(df_filtered)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)

# Teacher training logic

In [15]:
import random

# this is used because for a sequence generator training can slow down if
# an initial guess is wrong, the rest of the sequence is also most likely guessed wrong.
# hence this forced teaching approach is used to make training faster.


def train_step(model, encoder, decoder, optimizer, criterion, input_tensor, target_tensor, teacher_forcing_ratio=0.5):
    # input_tensor: [batch, input_len]
    # target_tensor: [batch, target_len]
    
    batch_size = input_tensor.size(0)
    target_len = target_tensor.size(1)
    vocab_size = decoder.out.out_features
    
    optimizer.zero_grad()
    
    # 1. Encode
    # LSTMs usually expect [seq_len, batch] if batch_first=False
    input_tensor = input_tensor.permute(1, 0) 
    encoder_outputs, (hidden, cell) = encoder(input_tensor)
    
    # 2. Decode step-by-step
    decoder_input = target_tensor[:, 0] # Start with <SOS>
    loss = 0
    
    for t in range(1, target_len):
        prediction, hidden, cell = decoder(decoder_input, hidden, cell, encoder_outputs)
        
        loss += criterion(prediction, target_tensor[:, t])
        
        # Decide whether to use real target or model's own prediction
        use_teacher_forcing = random.random() < teacher_forcing_ratio
        if use_teacher_forcing:
            decoder_input = target_tensor[:, t]
            # forcing the right answer
        else:
            decoder_input = prediction.argmax(1)
            # using the model prediction itself
            
    loss.backward()
    torch.nn.utils.clip_grad_norm_(encoder.parameters(), 1.0)
    torch.nn.utils.clip_grad_norm_(decoder.parameters(), 1.0)
    optimizer.step()
    
    return loss.item() / target_len

In [16]:
import gc
import torch

# this part is generated and untouched, just to clear the memory to avoid OOM

# Delete existing model variables if they exist
if 'enc' in locals(): del enc
if 'dec' in locals(): del dec
if 'optimizer' in locals(): del optimizer

# Standard garbage collection
gc.collect()

# Clear the PyTorch cache
torch.cuda.empty_cache()

# Optional: Print memory usage to see if it worked
print(f"Allocated: {torch.cuda.memory_allocated()/1024**2:.2f} MB")
print(f"Reserved: {torch.cuda.memory_reserved()/1024**2:.2f} MB")

Allocated: 0.00 MB
Reserved: 0.00 MB


# Training Loop

In [17]:
import torch.optim as optim
from tqdm import tqdm

# Hyperparameters (had to change cuz GPU was going OOM)
EMBEDDING_DIM = 128
HIDDEN_DIM = 512
N_LAYERS = 1
DROPOUT = 0.2
LEARNING_RATE = 0.001

# Initialize Model
attn = Attention(HIDDEN_DIM)
enc = Encoder(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT).cuda()
dec = Decoder(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, DROPOUT, attn).cuda()

optimizer = optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss(ignore_index=char_to_ix["<PAD>"])

def train(epochs):
    enc.train()
    dec.train()
    
    for epoch in range(epochs):
        total_loss = 0
        loop = tqdm(train_loader, leave=True)
        
        for i, (src, trg) in enumerate(loop):
            src, trg = src.cuda(), trg.cuda()
            
            # Use the train_step logic
            loss = train_step(None, enc, dec, optimizer, criterion, src, trg)
            total_loss += loss
            
            loop.set_description(f"Epoch [{epoch+1}/{epochs}]")
            loop.set_postfix(loss=loss)
            
        print(f"Epoch {epoch+1} Complete. Avg Loss: {total_loss/len(train_loader):.4f}")
        
        # Save checkpoint after every epoch
        torch.save(enc.state_dict(), f'encoder_ep{epoch+1}.pt')
        torch.save(dec.state_dict(), f'decoder_ep{epoch+1}.pt')

train(epochs=10)

/home/aakash/.conda/envs/gpu_env/lib/python3.10/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
Epoch [1/10]: 100%|████████████| 2710/2710 [44:58<00:00,  1.00it/s, loss=0.0639]


Epoch 1 Complete. Avg Loss: 0.4100


Epoch [2/10]: 100%|████████████| 2710/2710 [44:55<00:00,  1.01it/s, loss=0.0365]


Epoch 2 Complete. Avg Loss: 0.0684


Epoch [3/10]: 100%|████████████| 2710/2710 [44:56<00:00,  1.01it/s, loss=0.0188]


...

Epoch 8 Complete. Avg Loss: 0.0223


Epoch [9/10]: 100%|████████████| 2710/2710 [44:49<00:00,  1.01it/s, loss=0.0132]


Epoch 9 Complete. Avg Loss: 0.0200


Epoch [10/10]: 100%|███████████| 2710/2710 [44:58<00:00,  1.00it/s, loss=0.0155]

Epoch 10 Complete. Avg Loss: 0.0193
